# Bonus: phase ID and automatic quality control of XRD patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bashirkazimi/caden2026-ml-handson/blob/main/notebooks/02_bonus_spectra.ipynb)

**NFDI4ING summer school, IMD-2, Forschungszentrum Jülich. Bashir Kazimi (IAS-9)**

I assume that some of you worked with XRD patterns or Raman spectra in the practical part
of this summer school. This notebook transfers the ideas from the micrograph hands-on to
such 1D data:

1. **Phase identification**: which phase(s) is a pattern from?
2. **Intrinsic quality control**: can a model flag a *faulty* measurement (misaligned
   sample, too-short counting time, saturated detector, **wrong wavelength in the
   metadata**) before it pollutes your database? The NFDI4ING renewal proposal lists
   this as a planned task of IMD-2 in the task area Caden (measure C-4, "Intrinsic data
   quality control in research data management using AI",
   [zenodo.org/records/16793453](https://zenodo.org/records/16793453)).

We do not have your measurements here, so we **simulate** powder patterns from Bragg's law.
Peak positions come from the cubic lattice parameters of Si, YSZ and NiO (SOFC-relevant);
peak intensities are illustrative, not reference data. Part 4 shows how to plug in your
own exported files. Needs only numpy and matplotlib; runs in seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SEED = 0
rng = np.random.default_rng(SEED)
TWO_THETA = np.arange(20.0, 80.0, 0.02)  # degrees; the common grid every pattern is put on
WAVELENGTHS = {"Cu Ka": 1.5406, "Co Ka": 1.7890}  # Angstrom

## Part 1: Simulate measurements *with* metadata

Each phase is defined by its lattice parameter and allowed reflections. A "measurement"
adds what real instruments add: peak broadening (crystallite size), small strain shifts,
preferred orientation, a background and Poisson counting noise that depends on the
counting time. Every pattern comes with a metadata record, as it would in an ELN.

In [ ]:
def cubic_reflections(a, rule, hkl_max=4):
    """d-spacings (Angstrom) of allowed cubic reflections, with an illustrative intensity."""
    seen = {}
    for h in range(hkl_max + 1):
        for k in range(h + 1):
            for l in range(k + 1):
                if (h, k, l) == (0, 0, 0) or not rule(h, k, l):
                    continue
                s = h * h + k * k + l * l
                seen[s] = (a / np.sqrt(s), 1.0 / s**0.7)  # intensity: illustrative decay only
    return sorted(seen.values(), reverse=True)


def fcc(h, k, l):
    return len({h % 2, k % 2, l % 2}) == 1  # all even or all odd


def diamond(h, k, l):
    return fcc(h, k, l) and not (h % 2 == 0 and (h + k + l) % 4 == 2)


PHASES = {
    "Si": cubic_reflections(5.431, diamond),
    "YSZ": cubic_reflections(5.139, fcc),
    "NiO": cubic_reflections(4.177, fcc),
}
CLASSES = ["Si", "YSZ", "NiO", "NiO+YSZ"]  # NiO+YSZ: an SOFC anode cermet before reduction


def simulate(cls, wavelength=1.5406, count_time_s=1.0, zero_shift_deg=0.0, saturate=False,
             recorded_wavelength=None, rng=rng):
    phases = cls.split("+")
    weights = rng.dirichlet(np.ones(len(phases))) if len(phases) > 1 else [1.0]
    width = rng.uniform(0.08, 0.25)  # FWHM in degrees (crystallite size)
    y = np.zeros_like(TWO_THETA)
    for phase, w in zip(phases, weights):
        strain = rng.normal(0, 0.002)
        for d, intensity in PHASES[phase]:
            arg = wavelength / (2 * d * (1 + strain))
            if arg >= 1:
                continue
            pos = 2 * np.degrees(np.arcsin(arg)) + zero_shift_deg
            texture = rng.lognormal(0, 0.25)
            y += w * intensity * texture * np.exp(-4 * np.log(2) * (TWO_THETA - pos) ** 2 / width**2)
    y = 1000 * y / y.max()
    background = 60 + 40 * np.exp(-(TWO_THETA - 20) / 25) + rng.normal(0, 5)
    counts = rng.poisson((y + background) * count_time_s) / count_time_s
    if saturate:
        counts = np.minimum(counts, np.quantile(counts, 0.985))
    meta = {"phase_label": cls, "wavelength_A": recorded_wavelength or wavelength,
            "count_time_s": count_time_s, "step_deg": 0.02, "instrument": "simulated"}
    return counts, meta


def normalise(counts):
    """Remove the scale and background so patterns from different runs are comparable."""
    c = counts - np.percentile(counts, 10)
    return c / (np.linalg.norm(c) + 1e-9)


X, Y, META = [], [], []
for label, cls in enumerate(CLASSES):
    for _ in range(60):
        counts, meta = simulate(cls, count_time_s=rng.uniform(0.5, 2.0))
        X.append(normalise(counts))
        Y.append(label)
        META.append(meta)
X, Y = np.array(X), np.array(Y)
print(f"{len(X)} simulated patterns, {X.shape[1]} points each; example metadata:\n  {META[0]}")

fig, ax = plt.subplots(figsize=(11, 3.5))
for label, cls in enumerate(CLASSES):
    ax.plot(TWO_THETA, X[Y == label][0] + 0.15 * label, lw=0.8, label=cls)
ax.set_xlabel("2θ (°), Cu Kα")
ax.set_yticks([])
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

## Part 2: Phase identification with PCA + nearest centroid

Principal component analysis (PCA) compresses each 3000-point pattern into a handful of
numbers that capture most of the variation. A new pattern is assigned to the class whose
average (centroid) is closest in that space. Same rules as before: **fit on training
data only**, evaluate on held-out patterns.

In [ ]:
order = rng.permutation(len(X))
train, test = order[: int(0.7 * len(X))], order[int(0.7 * len(X)):]

mean = X[train].mean(axis=0)
_, sv, components = np.linalg.svd(X[train] - mean, full_matrices=False)
N_COMPONENTS = 15


def project(x):
    return (x - mean) @ components[:N_COMPONENTS].T


Z = project(X)
centroids = np.array([Z[train][Y[train] == k].mean(axis=0) for k in range(len(CLASSES))])
pred = np.argmin(((Z[test][:, None, :] - centroids[None]) ** 2).sum(-1), axis=1)
confusion = np.zeros((len(CLASSES), len(CLASSES)), int)
for t, p in zip(Y[test], pred):
    confusion[t, p] += 1
print(f"test accuracy: {(pred == Y[test]).mean():.1%}  (explained variance by {N_COMPONENTS} PCs: "
      f"{(sv[:N_COMPONENTS] ** 2).sum() / (sv ** 2).sum():.1%})")
print("confusion matrix (rows = true, columns = predicted):")
print(" " * 9 + "".join(f"{c:>9}" for c in CLASSES))
for k, row in enumerate(confusion):
    print(f"{CLASSES[k]:>9}" + "".join(f"{v:>9}" for v in row))

## Part 3: Intrinsic quality control: flag faulty measurements

Train PCA on **good** measurements only. A new pattern that the model cannot reconstruct
well (large *reconstruction error*) does not look like anything it has seen → flag it for
a human to check. We now simulate four typical faults:

| fault | cause in the lab |
|---|---|
| zero shift | sample height misaligned → all peaks shifted |
| short count | measurement stopped too early → noisy |
| saturation | detector saturated → flattened peak tops |
| wrong wavelength | measured with Co Kα, **metadata says Cu Kα** |

In [ ]:
def recon_error(x):
    z = (x - mean) @ components[:N_COMPONENTS].T
    return np.linalg.norm(x - mean - z @ components[:N_COMPONENTS], axis=-1)


good_err = recon_error(X[test])
threshold = np.quantile(recon_error(X[train]), 0.99)

faults = {
    "zero shift": dict(zero_shift_deg=0.4),
    "short count": dict(count_time_s=0.02),
    "saturation": dict(saturate=True),
    "wrong wavelength": dict(wavelength=WAVELENGTHS["Co Ka"], recorded_wavelength=WAVELENGTHS["Cu Ka"]),
}
fault_err = {}
for name, kwargs in faults.items():
    fault_err[name] = np.array([recon_error(normalise(simulate(CLASSES[rng.integers(4)], **kwargs)[0]))
                                for _ in range(30)])

print(f"threshold (99th percentile of good training data): {threshold:.3f}")
print(f"good test patterns flagged: {(good_err > threshold).mean():.0%}")
for name, err in fault_err.items():
    print(f"{name:>17}: flagged {(err > threshold).mean():.0%}")

fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0, max(e.max() for e in fault_err.values()) * 1.05, 60)
ax.hist(good_err, bins=bins, alpha=0.6, label="good (test)")
for name, err in fault_err.items():
    ax.hist(err, bins=bins, alpha=0.5, label=name)
ax.axvline(threshold, color="k", ls="--", lw=1, label="flag threshold")
ax.set_xlabel("reconstruction error")
ax.set_ylabel("count")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Discussion

1. Which faults are easy to catch, which are hard? Why might a small zero shift slip
   through?
2. The "wrong wavelength" pattern is a *perfectly good measurement* with *wrong
   metadata*. What would happen to a phase-ID model, or to a lattice-parameter fit, that
   trusts the metadata?
3. Which fields would an ELN template (for example in eLabFTW) need to record so that
   such checks can run automatically when a measurement is uploaded?
4. Look back at the confusion matrix of Part 2: why is the NiO+YSZ mixture the hardest
   class for a nearest-centroid classifier?

*Notes:*

## Part 4: Plug in your own data

If you have XRD patterns of your own, put them on the same 2θ grid. The code below
assumes two-column text files (2θ, intensity); I do not know what your instrument or ELN
exports, so adapt the loading line to your actual file format:

```python
from pathlib import Path
own = {}
for path in sorted(Path("my_xrd").glob("*.xy")):
    tth, counts = np.loadtxt(path, unpack=True)          # adapt delimiter/skiprows
    own[path.stem] = normalise(np.interp(TWO_THETA, tth, counts))
for name, x in own.items():
    print(f"{name:>20}: reconstruction error {recon_error(x):.3f}",
          "FLAG" if recon_error(x) > threshold else "")
```

A model trained on simulated data will *not* transfer to real data without care
(instrument function, background, fluorescence…), which is exactly why the
instrument metadata matter.